# Qwen 3.5 9B | Arabic Agentic Coding Assistant (Cloud Stable)

This variant keeps the explicit agentic quality targets for Qwen 3.5 9B Native 16-bit, but explicitly mirrors the strict custom AMD OneClick Notebook Environment limits. 
It establishes a complete python `venv` targeting `rocm`, completely bypasses `unsloth[colab-new]`, and utilizes aggressive step-chunking to survive 50-minute kernel timeouts.

### 1. Robust AMD Cloud Installation

In [ ]:
# === Create a clean AMD/Unsloth kernel ===
import os, re, shutil, subprocess, sys
from pathlib import Path
INSTALL_ROOT = os.environ.get('UNSLOTH_INSTALL_ROOT', str(Path.home() / 'unsloth-qwen'))
VENV = os.path.join(INSTALL_ROOT, 'venv')
TMPDIR = os.path.join(INSTALL_ROOT, 'tmp')
PIP_CACHE_DIR = os.path.join(INSTALL_ROOT, 'pip-cache')
PY = os.path.join(VENV, 'bin', 'python')
PIP = [PY, '-m', 'pip']

def run(cmd):
    print('\n$ ' + ' '.join(cmd))
    env = os.environ.copy()
    env['TMPDIR'] = TMPDIR
    env['TEMP'] = TMPDIR
    env['TMP'] = TMPDIR
    env['PIP_CACHE_DIR'] = PIP_CACHE_DIR
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.stdout.strip():
        out = r.stdout.strip().splitlines()
        for line in out[-8:]:
            print(line)
    if r.returncode != 0:
        err = r.stderr.strip() or '(no stderr)'
        raise RuntimeError(err)

def detect_rocm_tag():
    candidates = []
    try:
        r = subprocess.run(['amd-smi', 'version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    try:
        candidates.append(Path('/opt/rocm/.info/version').read_text())
    except Exception:
        pass
    try:
        r = subprocess.run(['hipconfig', '--version'], capture_output=True, text=True)
        candidates.append(r.stdout)
    except FileNotFoundError:
        pass
    for text in candidates:
        m = re.search(r'(?:ROCm version: |HIP version: )([0-9]+)\.([0-9]+)', text)
        if not m:
            m = re.search(r'([0-9]+)\.([0-9]+)', text)
        if m:
            return f'rocm{m.group(1)}.{m.group(2)}'
    return 'rocm7.0'

Path(INSTALL_ROOT).mkdir(parents=True, exist_ok=True)
Path(TMPDIR).mkdir(parents=True, exist_ok=True)
Path(PIP_CACHE_DIR).mkdir(parents=True, exist_ok=True)
rocm_tag = detect_rocm_tag()
print('ROCm wheel tag:', rocm_tag)
print('Install root:', INSTALL_ROOT)

# Delete VENV strictly if re-running
if os.path.isdir(VENV):
    shutil.rmtree(VENV)
run([sys.executable, '-m', 'venv', VENV])
run(PIP + ['install', '--upgrade', 'pip', 'setuptools', 'wheel', 'ipykernel'])
run(PIP + ['install', '--upgrade', '--force-reinstall', 'torch', 'torchvision', 'torchaudio', '--index-url', f'https://download.pytorch.org/whl/{rocm_tag}'])
run(PIP + ['install', '--no-deps', 'unsloth', 'unsloth-zoo'])
run(PIP + ['install', '--no-deps', 'git+https://github.com/unslothai/unsloth-zoo.git'])
run(PIP + ['install', 'unsloth[amd] @ git+https://github.com/unslothai/unsloth'])
run(PIP + ['install', '--upgrade', 'git+https://github.com/huggingface/transformers.git'])
run(PIP + ['install', '--upgrade', 'timm', 'torchcodec'])
run([PY, '-m', 'ipykernel', 'install', '--user', '--name', 'unsloth-qwen-amd', '--display-name', 'Python (unsloth-qwen-amd)'])

print('\n\u2705 Kernel created: Python (unsloth-qwen-amd)')
print('Next: Kernel -> Change Kernel -> Python (unsloth-qwen-amd)')
print('Then restart kernel and run the next cell.')

In [ ]:
# === Verify you are on the clean Unsloth AMD kernel ===
import os, sys
from pathlib import Path
VENV = os.environ.get('UNSLOTH_VENV', str(Path.home() / 'unsloth-qwen' / 'venv'))
on_expected_kernel = (
    sys.prefix == VENV
    or sys.executable.startswith(VENV + os.sep)
    or os.environ.get('VIRTUAL_ENV') == VENV
)
if not on_expected_kernel:
    raise RuntimeError('Wrong kernel. Change kernel to Python (unsloth-qwen-amd), restart, and re-run.')

import unsloth, unsloth_zoo, huggingface_hub, regex, transformers
print(f'unsloth: {getattr(unsloth, "__version__", "OK")}')


In [ ]:
import os
import huggingface_hub

HF_TOKEN = ""
HF_REPO_ID = "mdtita/Qwen3.5-9B-Arabic-Agent-SFT"

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN or HF_TOKEN == "YOUR_HF_TOKEN":
    print("\u26a0\ufe0f No HF token set. Hub pushing disabled!")
    HF_TOKEN = ""
    PUSH_TO_HUB = False
else:
    print("\u2705 Hugging Face Token recognized!")
    huggingface_hub.login(token=HF_TOKEN)
    PUSH_TO_HUB = True

TRAINING_OUTPUT_DIR = os.environ.get("TRAINING_OUTPUT_DIR", str(Path.home() / "outputs_qwen35_9B_arabic_sft"))
Path(TRAINING_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

### 2. Model Diagnostics (Qwen 9B Native)

In [ ]:
from unsloth import FastLanguageModel
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:512"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

print("=== Loading Qwen 3.5 9B Model ===")
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.5-9B",   # Explicit namespace (Unsloth unquant isn't published)
    max_seq_length = max_seq_length,
    load_in_4bit = False,             # LESSON: Qwen 3.5 crashes on 4-bit
    load_in_16bit = True,             # LESSON: Native ROCm bf16 limits
    fast_inference = False,
)

print("=== Adding LoRA Adapters ===")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 3. Load & Process Validated Formats

In [ ]:
from datasets import load_dataset
print("=== Loading Curated Arabic Agentic Dataset ===")
dataset = load_dataset('mtita/qwen3.5-arabic-agent-dataset', split='train')
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
    mapping = {"role": "role", "content": "content", "user": "user", "assistant": "assistant", "system": "system"}
)
def formatting_prompts_func(examples):
    texts = []
    for msgs in examples["messages"]:
        texts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return tokenizer(text=texts, truncation=True, max_length=max_seq_length)
dataset = dataset.map(
    formatting_prompts_func, 
    batched=True, 
    remove_columns=dataset.column_names,
    num_proc=16
)


### 4. Crash-Resilient Chunk Training

Handles 50m VM kernel timeouts by slicing epochs into strict saved/pushed iterations.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    max_seq_length = max_seq_length,
    dataset_num_proc = 16,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 20,
        learning_rate = 2e-5,
        num_train_epochs = 2,
        logging_steps = 5,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = TRAINING_OUTPUT_DIR,
        save_strategy = "steps",
        save_steps = 400,
        save_total_limit = 2,
        report_to = "none",
        push_to_hub = False, # OFF: explicitly overriding unreliable background thread
        bf16 = True,
    ),
)

In [ ]:
import gc
import json
from huggingface_hub import snapshot_download, HfApi
from transformers.trainer_utils import get_last_checkpoint

def get_step_from_checkpoint(ckpt_dir):
    if not ckpt_dir: return 0
    try:
        with open(os.path.join(ckpt_dir, "trainer_state.json")) as f:
            return json.load(f).get("global_step", 0)
    except:
        try: return int(ckpt_dir.split("-")[-1])
        except: return 0

# 1. Re-sync from Hub if VM reset
if PUSH_TO_HUB:
    print(f"\n\ud83d\udce5 Explicitly fetching rescues from {HF_REPO_ID}...")
    try:
        snapshot_download(repo_id=HF_REPO_ID, local_dir=TRAINING_OUTPUT_DIR, token=HF_TOKEN, allow_patterns=["*checkpoint*"])
    except Exception as e:
        print("Clean start.")

CHUNK_SIZE = 400
TOTAL_CHUNKS = 40

for i in range(TOTAL_CHUNKS):
    LAST_CHECKPOINT = get_last_checkpoint(TRAINING_OUTPUT_DIR)
    current_max_steps = (i + 1) * CHUNK_SIZE
    trainer.args.max_steps = current_max_steps
    
    completed_steps = get_step_from_checkpoint(LAST_CHECKPOINT)
    if LAST_CHECKPOINT and completed_steps >= current_max_steps:
        print(f"\u23ed\ufe0f Skipping chunk {i+1} (Done)")
        continue
        
    print(f"\n\ud83d\ude80 Running Chunk {i+1} to step {current_max_steps}...")
    
    try:
        stats = trainer.train(resume_from_checkpoint=LAST_CHECKPOINT)
    except Exception as e:
        print(f"Timeout/Interrupt saved: {e}")
        break
        
    if PUSH_TO_HUB:
        print(f"\ud83d\udce6 Force-uploading chunk {i+1}...")
        api = HfApi(token=HF_TOKEN)
        api.upload_folder(
            folder_path=TRAINING_OUTPUT_DIR, repo_id=HF_REPO_ID, repo_type="model",
            allow_patterns=["checkpoint-*/*"], commit_message=f"Sync step {current_max_steps}"
        )
    
    gc.collect()
    torch.cuda.empty_cache()


### 5. Final Export & GGUF Compilation

In [ ]:
model.save_pretrained("qwen35_9b_lora_final")
tokenizer.save_pretrained("qwen35_9b_lora_final")

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
    print("\u2728 Final Weights Synced to Hub!")
    
# Export to Q4_K_M GGUF format automatically for local deployment
try:
    model.save_pretrained_gguf(
        "qwen35_9b_agent_gguf",
        tokenizer,
        quantization_method="q4_k_m"
    )
    if PUSH_TO_HUB:
        model.push_to_hub_gguf(
            HF_REPO_ID + "-GGUF",
            tokenizer,
            quantization_method="q4_k_m",
            token=HF_TOKEN,
        )
except Exception as e:
    print(f"GGUF Compilation Failed: {e}")